# Практика 6. Ансамбли — версия с решениями

> Этот файл предназначен преподавателю: все задания заполнены, добавлены комментарии к разбору. Студентам выдаётся версия без суффикса `_solutions`.

# Практика 6. Ансамбли

**Курс «Машинное обучение» · Часть 1, занятие 6**

Сегодня работаем с реальными данными — ценами на жильё в Калифорнии — и смотрим, сколько даёт переход от одного дерева к ансамблю.

План:

1. Данные и точка отсчёта
2. Одно дерево: откуда берётся разброс
3. Бэггинг своими руками — чтобы увидеть механизм
4. Случайный лес и out-of-bag оценка
5. Градиентный бустинг и ранняя остановка
6. Задание: скорость обучения и число деревьев
7. Подбор параметров через Optuna
8. Сравнение библиотек и стекинг

> В Colab понадобится Optuna:
> `!pip install -q optuna`

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
)
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Сюда по ходу занятия складываем качество каждой модели,
# чтобы в конце сравнить всё одним графиком
scores = {}
print("готово")

---
## 1. Данные и точка отсчёта

Датасет California housing: 20 640 районов Калифорнии, по каждому известны медианный доход жителей, возраст застройки, число комнат, население и координаты. Предсказываем медианную стоимость дома в районе.

Целевую переменную сразу переведём в **тысячи долларов** — так числа читаются глазами, а ошибка измеряется в деньгах.

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
features = list(data.feature_names)

X = df[features].values
y = df["MedHouseVal"].values * 100      # в тысячах долларов

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

print(f"обучающая: {X_train.shape}, тестовая: {X_test.shape}")
print(f"признаки: {', '.join(features)}")
print()
print(f"медианная цена: {np.median(y):.1f} тыс. $")
print(f"разброс:        от {y.min():.1f} до {y.max():.1f} тыс. $")
print(f"объектов на верхней границе (500 тыс.): {(y >= 500).sum()} — "
      f"{(y >= 500).mean():.1%} выборки")

Последняя строка — особенность реальных данных: цены **обрезаны сверху**. Всё, что дороже 500 тысяч, записано как ровно 500. Такие объекты модель предсказать точно не сможет, и на них будет приходиться заметная часть ошибки. Это стоит знать заранее, иначе потом будешь искать проблему в модели.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(y, bins=60, color="#2B5CE6", alpha=0.85)
axes[0].axvline(500, ls="--", color="#C0393B")
axes[0].text(470, axes[0].get_ylim()[1] * 0.8, "обрезка\nсверху",
             ha="right", color="#C0393B")
axes[0].set_xlabel("цена, тыс. $")
axes[0].set_ylabel("число районов")
axes[0].set_title("Распределение цены")

sc = axes[1].scatter(df["Longitude"], df["Latitude"], c=y, s=4,
                     cmap="viridis", alpha=0.6)
axes[1].set_xlabel("долгота")
axes[1].set_ylabel("широта")
axes[1].set_title("Цена на карте Калифорнии")
plt.colorbar(sc, ax=axes[1], label="цена, тыс. $")

plt.tight_layout()
plt.show()

print("На карте видны два дорогих пятна — Сан-Франциско и Лос-Анджелес.")
print("То есть в данных есть сильная пространственная структура, и дерево")
print("умеет её ловить: достаточно порогов по широте и долготе.")

In [ ]:
# Точка отсчёта: модель, которая всегда называет медианную цену
dummy = DummyRegressor(strategy="median").fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

mae_dummy = mean_absolute_error(y_test, pred_dummy)
scores["константа"] = mae_dummy

print(f"MAE  константной модели: {mae_dummy:.2f} тыс. $")
print(f"MAPE константной модели: "
      f"{mean_absolute_percentage_error(y_test, pred_dummy):.1%}")
print()
print("Любая модель обязана быть лучше этого. MAE здесь удобнее: она")
print("измеряется в тысячах долларов, и её можно показать заказчику.")

---
## 2. Одно дерево

Обучим дерево без ограничений и дерево глубины 8 — и посмотрим на разрыв между обучением и тестом.

In [ ]:
tree_full = DecisionTreeRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
tree_8 = DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE).fit(X_train, y_train)

mae_tree = mean_absolute_error(y_test, tree_full.predict(X_test))
scores["одно дерево"] = mae_tree

print(f"дерево без ограничений: глубина {tree_full.get_depth()}, "
      f"листьев {tree_full.get_n_leaves()}")
print()
print(f"{'модель':26} {'MAE обучение':>13} {'MAE тест':>10}")
print(f"{'без ограничений':26} "
      f"{mean_absolute_error(y_train, tree_full.predict(X_train)):13.2f} "
      f"{mae_tree:10.2f}")
print(f"{'глубина 8':26} "
      f"{mean_absolute_error(y_train, tree_8.predict(X_train)):13.2f} "
      f"{mean_absolute_error(y_test, tree_8.predict(X_test)):10.2f}")
print()
print("Ноль на обучении — дерево довело ветви до отдельных районов")
print("и запомнило их цены. На тесте от этого никакой пользы.")

Дерево вдвое точнее константы, но посмотрите на разрыв: ноль на обучении и больше сорока тысяч долларов на тесте. Ограничение глубины почти ничего не меняет. Классическая картина: **маленькое смещение, огромный разброс** — дерево подстраивается под каждый район обучающей выборки, а на новых районах эти подстройки не работают.

С таким деревом и работает бэггинг: усреднять надо именно переобученные модели.

---
## 3. Бэггинг своими руками

Механизм в три шага: берём бутстрап-выборку, обучаем на ней дерево, усредняем предсказания. Напишем это руками — там нет ничего, кроме этих трёх шагов.

In [ ]:
class MyBagging:
    """Бэггинг над произвольным регрессором.

    Каждая базовая модель учится на своей бутстрап-выборке — n объектов,
    взятых с возвращением. Предсказание ансамбля — среднее предсказаний.
    """

    def __init__(self, base_estimator, n_estimators=50, random_state=42):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n = len(X)
        self.models_ = []
        self.oob_indices_ = []

        for _ in range(self.n_estimators):
            idx = rng.integers(0, n, size=n)          # с возвращением
            oob = np.setdiff1d(np.arange(n), idx)     # что не попало
            self.models_.append(clone(self.base_estimator).fit(X[idx], y[idx]))
            self.oob_indices_.append(oob)
        return self

    def predict(self, X):
        return np.array([m.predict(X) for m in self.models_]).mean(axis=0)

    def oob_mae(self, X, y):
        """Ошибка на объектах, не попавших в бутстрап каждой модели."""
        total = np.zeros(len(X))
        count = np.zeros(len(X))
        for model, oob in zip(self.models_, self.oob_indices_):
            total[oob] += model.predict(X[oob])
            count[oob] += 1
        seen = count > 0
        return mean_absolute_error(y[seen], total[seen] / count[seen])


bag = MyBagging(DecisionTreeRegressor(random_state=RANDOM_STATE),
                n_estimators=50, random_state=RANDOM_STATE).fit(X_train, y_train)

mae_bag = mean_absolute_error(y_test, bag.predict(X_test))
scores["бэггинг 50 деревьев"] = mae_bag

print(f"одно дерево:        MAE {mae_tree:.2f}")
print(f"бэггинг 50 деревьев: MAE {mae_bag:.2f}")
print(f"out-of-bag оценка:   MAE {bag.oob_mae(X_train, y_train):.2f}")
print()
print("Оценка out-of-bag получена без отложенной выборки: каждый объект")
print("проверен теми деревьями, которым он при бутстрапе не достался.")

In [ ]:
# Главный вопрос: ансамбль лучше своих участников — или лучшего из них?
single_maes = [mean_absolute_error(y_test, m.predict(X_test)) for m in bag.models_]

sizes = [1, 2, 5, 10, 20, 40, 80]
curve = []
for n in sizes:
    b = MyBagging(DecisionTreeRegressor(random_state=RANDOM_STATE),
                  n_estimators=n, random_state=RANDOM_STATE).fit(X_train, y_train)
    curve.append(mean_absolute_error(y_test, b.predict(X_test)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(sizes, curve, marker="o", color="#2B5CE6")
axes[0].axhline(mae_tree, ls="--", color="#C0393B", label="одно дерево")
axes[0].set_xscale("log")
axes[0].set_xticks(sizes)
axes[0].set_xticklabels(sizes)
axes[0].set_xlabel("деревьев в ансамбле")
axes[0].set_ylabel("MAE на тесте, тыс. $")
axes[0].set_title("Усреднение снижает ошибку и выходит на плато")
axes[0].legend()

axes[1].hist(single_maes, bins=16, color="#6B7484", alpha=0.8,
             label="отдельные деревья ансамбля")
axes[1].axvline(mae_bag, color="#2B5CE6", lw=2.5,
                label=f"ансамбль из них: {mae_bag:.1f}")
axes[1].set_xlabel("MAE на тесте, тыс. $")
axes[1].set_ylabel("число деревьев")
axes[1].set_title("Ансамбль лучше любого своего участника")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"деревья внутри ансамбля: MAE от {min(single_maes):.1f} до "
      f"{max(single_maes):.1f}, в среднем {np.mean(single_maes):.2f}")
print(f"ансамбль из этих же деревьев:  MAE {mae_bag:.2f}")

Правая картинка — то, из-за чего бэггинг вообще работает. Ни одно дерево внутри ансамбля не дотягивает до его результата: каждое ошибается в среднем сильнее, чем они все вместе.

Причина в том, что ошибки деревьев **направлены в разные стороны**. Каждое дерево видело свою бутстрап-выборку, поэтому переобучилось по-своему: где одно завысило цену, другое занижает. При усреднении случайная часть ошибки гасится, а общая закономерность остаётся.

Левая картинка добавляет практический вывод: основной выигрыш приходит на первых десяти деревьях, дальше кривая ложится на плато. Лишние деревья в бэггинге не вредят — просто перестают помогать.

### Вопрос

Бэггинг над глубокими деревьями снизил ошибку с 46 до 33. Что будет, если усреднять не глубокие деревья, а пни — деревья глубины 1?

**Что хотелось услышать:**

Почти ничего не изменится. У пня огромное смещение: одним разбиением цену жилья не описать, и все пни на разных бутстрап-выборках получатся почти одинаковыми. Усреднять нечего — разброс у них и так мал, а смещение усреднение не лечит. Отсюда правило: в бэггинг берут переобученные модели, в бустинг — слабые.

---
## 4. Случайный лес

Лес отличается от бэггинга одной деталью: в каждом узле дерево выбирает разбиение не из всех признаков, а из случайного подмножества. Деревья получаются ещё менее похожими друг на друга, и усреднение работает лучше.

In [ ]:
forest = RandomForestRegressor(
    n_estimators=300, oob_score=True, n_jobs=-1, random_state=RANDOM_STATE
)
start = time.perf_counter()
forest.fit(X_train, y_train)
forest_time = time.perf_counter() - start

mae_forest = mean_absolute_error(y_test, forest.predict(X_test))
scores["случайный лес"] = mae_forest

print(f"случайный лес (300 деревьев): MAE {mae_forest:.2f}, "
      f"обучение {forest_time:.1f} с")
print(f"наш бэггинг (50 деревьев):    MAE {mae_bag:.2f}")
print(f"out-of-bag R²:                {forest.oob_score_:.4f}")

order = np.argsort(forest.feature_importances_)
plt.figure(figsize=(9, 4))
plt.barh(np.array(features)[order], forest.feature_importances_[order],
         color="#2B5CE6", alpha=0.85)
plt.xlabel("важность признака")
plt.title("Что лес считает главным")
plt.tight_layout()
plt.show()

print("Доход жителей с большим отрывом, за ним заселённость жилья.")
print("Широта и долгота по отдельности скромные, но вместе дают больше,")
print("чем любой признак кроме дохода, — как и обещала карта.")
print("Важность здесь встроенная: она считается по тому, сколько раз признак")
print("использовался в разбиениях и как снижал ошибку. Честнее её мерить")
print("перестановками, но это тема лекции 8.")

---
## 5. Градиентный бустинг и ранняя остановка

Бустинг строит деревья по очереди: каждое следующее учится на том, что не смогли предыдущие. Поэтому лишние деревья здесь **вредят** — проверим это, обучив модель с крупным шагом заведомо дольше, чем нужно.

In [ ]:
# Сначала посмотрим, что будет без остановки. Терпение больше числа
# деревьев — остановка не сработает, но кривую на отложенной части
# модель всё равно запишет
no_stop = HistGradientBoostingRegressor(
    max_iter=1500, learning_rate=0.3,
    early_stopping=True, n_iter_no_change=1500, validation_fraction=0.2,
    scoring="neg_mean_absolute_error", random_state=RANDOM_STATE,
).fit(X_train, y_train)

# Кривые хранятся в единицах метрики, то есть со знаком минус
train_curve = -np.array(no_stop.train_score_)
valid_curve = -np.array(no_stop.validation_score_)
best_iter = int(np.argmin(valid_curve))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(train_curve, label="обучающая часть", color="#2B5CE6")
axes[0].plot(valid_curve, label="отложенная часть", color="#C0393B")
axes[0].set_ylim(0, 60)
axes[0].set_xlabel("число деревьев")
axes[0].set_ylabel("MAE, тыс. $")
axes[0].set_title("Кривые расходятся")
axes[0].legend()

tail = slice(50, None)
axes[1].plot(np.arange(len(valid_curve))[tail], valid_curve[tail], color="#C0393B")
axes[1].axvline(best_iter, ls="--", color="#6B7484",
                label=f"минимум: {best_iter}-е дерево")
axes[1].set_xlabel("число деревьев")
axes[1].set_ylabel("MAE на отложенной части, тыс. $")
axes[1].set_title("Отложенная часть крупно: после минимума ошибка растёт")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"на обучающей части ошибка упала до {train_curve[-1]:.1f} тыс. $")
print(f"на отложенной: минимум {valid_curve[best_iter]:.2f} на {best_iter}-м дереве,")
print(f"к {len(valid_curve) - 1}-му выросла до {valid_curve[-1]:.2f} — "
      f"на {valid_curve[-1] - valid_curve[best_iter]:.1f} тыс. $")

Синяя кривая падает до конца: ансамбль всё точнее описывает обучающую выборку. Красная после минимума ползёт вверх — новые деревья подгоняются под шум и делают прогноз на новых данных хуже. В бэггинге такого не бывает: там деревья независимы, а здесь каждое следующее исправляет ошибки предыдущих, в том числе случайные.

Нужное число деревьев не угадывают, а находят **ранней остановкой**: модель сама откладывает часть обучающей выборки и строит деревья, пока качество на ней улучшается.

In [ ]:
boost = HistGradientBoostingRegressor(
    max_iter=2000,                         # верхняя граница, а не план
    learning_rate=0.1,
    early_stopping=True,
    n_iter_no_change=50,                   # терпение: столько итераций без улучшения
    validation_fraction=0.2,
    scoring="neg_mean_absolute_error",
    random_state=RANDOM_STATE,
)
start = time.perf_counter()
boost.fit(X_train, y_train)
boost_time = time.perf_counter() - start

mae_boost = mean_absolute_error(y_test, boost.predict(X_test))
scores["бустинг"] = mae_boost

print(f"задано деревьев: 2000, обучено: {boost.n_iter_}")
print(f"MAE на тесте: {mae_boost:.2f}, обучение {boost_time:.1f} с")

**Терпение** (`n_iter_no_change=50`) нужно потому, что кривая на отложенной части не падает монотонно — она шумит. Если останавливаться на первом же ухудшении, обучение прервётся слишком рано.

### Задание

Скорость обучения `learning_rate` и число деревьев связаны: чем меньше шаг, тем больше шагов нужно. Проверим это измерением.

1. Для каждого значения `learning_rate` из списка обучите бустинг **с ранней остановкой**
2. Запишите, сколько деревьев понадобилось и какое качество получилось
3. По графику скажите, подтверждается ли правило «уменьшили скорость вдвое — вдвое больше деревьев»

Результаты сложите в таблицу `table_rates` со столбцами `learning_rate`, `деревьев`, `MAE`.

> **Что посмотреть:** модель с ранней остановкой собрана в ячейке выше — `HistGradientBoostingRegressor(max_iter=..., learning_rate=..., early_stopping=True, n_iter_no_change=50, validation_fraction=0.2, scoring="neg_mean_absolute_error", random_state=RANDOM_STATE)`. После обучения число деревьев лежит в `n_iter_`. Возьмите `max_iter=3000`, иначе при малой скорости обучение упрётся в потолок.

In [ ]:
rates = [0.3, 0.1, 0.05, 0.02]
rows = []

for lr in rates:
    start = time.perf_counter()

    model = HistGradientBoostingRegressor(
        max_iter=3000, learning_rate=lr,
        early_stopping=True, n_iter_no_change=50, validation_fraction=0.2,
        scoring="neg_mean_absolute_error", random_state=RANDOM_STATE,
    ).fit(X_train, y_train)

    elapsed = time.perf_counter() - start
    mae = mean_absolute_error(y_test, model.predict(X_test))
    rows.append({"learning_rate": lr, "деревьев": model.n_iter_,
                 "MAE": round(mae, 2), "время, с": round(elapsed, 1)})
    print(f"lr={lr:<5} деревьев {model.n_iter_:5}   MAE {mae:6.2f}   {elapsed:5.1f} с")

table_rates = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(table_rates["learning_rate"], table_rates["деревьев"],
             marker="o", color="#2B5CE6")
axes[0].set_xscale("log")
axes[0].set_xticks(rates, [str(r) for r in rates])
axes[0].minorticks_off()
axes[0].set_xlabel("learning_rate"); axes[0].set_ylabel("деревьев до остановки")
axes[0].set_title("Меньше шаг — больше деревьев")

axes[1].plot(table_rates["learning_rate"], table_rates["MAE"],
             marker="o", color="#C0393B")
axes[1].set_xscale("log")
axes[1].set_xticks(rates, [str(r) for r in rates])
axes[1].minorticks_off()
axes[1].set_xlabel("learning_rate"); axes[1].set_ylabel("MAE на тесте, тыс. $")
axes[1].set_title("Крупный шаг хуже, дальше — плато")

plt.tight_layout()
plt.show()

print()
print("Зависимость обратная, но не строго пропорциональная: при шаге 0.3")
print("хватает двух сотен деревьев, при 0.02 нужно больше полутора тысяч.")
print()
print("С качеством иначе: шаг 0.3 заметно хуже остальных, а начиная с 0.1")
print("ошибка выходит на плато — разница между 0.1, 0.05 и 0.02 в пределах")
print("пары сотен долларов. Время обучения при этом различается в разы.")
print("Поэтому шаг выбирают из соображений бюджета: 0.1 для экспериментов,")
print("0.05 для финальной модели, если есть время её обучить.")


In [ ]:
# Проверка задания
trees = table_rates.set_index("learning_rate")["деревьев"]
assert len(table_rates) == 4, f"строк в таблице {len(table_rates)}, а ожидалось четыре"
assert trees[0.02] > trees[0.3],     "при меньшей скорости обучения деревьев должно потребоваться больше"
assert trees[0.02] > 3 * trees[0.3],     "при уменьшении шага в 15 раз число деревьев обязано вырасти в разы"
assert trees.max() < 3000,     "обучение упёрлось в max_iter — ранняя остановка не сработала"
assert table_rates["MAE"].min() < 31,     f"качество хуже ожидаемого: {table_rates['MAE'].tolist()}"
print("Задание — верно ✓")

> **Для преподавателя.** Если у кого-то числа деревьев разойдутся с соседними на десятки — это нормально: ранняя остановка зависит от внутреннего разбиения. Важен порядок величины и то, что MAE при всех скоростях почти одинаковая.

---
## 6. Подбор параметров через Optuna

У бустинга ручек больше, чем у дерева, и сетка по ним разрастается мгновенно. `GridSearchCV` из прошлого занятия здесь уже не помощник: четыре параметра по пять значений — это 625 обучений.

Optuna работает иначе. Она не перебирает заранее заданные точки, а **выбирает следующую по результатам предыдущих**: смотрит, где получалось хорошо, и пробует рядом. Нужно описать не список значений, а диапазон для каждого параметра.

Каждую пробу оцениваем кросс-валидацией на обучающей выборке, как на прошлом занятии. Тридцать проб — самая долгая ячейка занятия.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

cv3 = KFold(3, shuffle=True, random_state=RANDOM_STATE)

def make_boost(**params):
    return HistGradientBoostingRegressor(
        max_iter=1000, early_stopping=True, n_iter_no_change=30,
        validation_fraction=0.15, scoring="neg_mean_absolute_error",
        random_state=RANDOM_STATE, **params,
    )

def cv_mae(model):
    """MAE на кросс-валидации по обучающей выборке. Тест не участвует."""
    return -cross_val_score(model, X_train, y_train, cv=cv3,
                            scoring="neg_mean_absolute_error", n_jobs=-1).mean()

def objective(trial):
    """Одна проба: Optuna предлагает параметры, мы возвращаем ошибку."""
    return cv_mae(make_boost(
        learning_rate=trial.suggest_float("learning_rate", 0.03, 0.3, log=True),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 8, 128, log=True),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 100, log=True),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-3, 10.0, log=True),
    ))

study = optuna.create_study(
    direction="minimize",                                  # MAE надо уменьшать
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE), # для воспроизводимости
)
start = time.perf_counter()
study.optimize(objective, n_trials=30)
search_time = time.perf_counter() - start

default_cv = cv_mae(make_boost())

print(f"30 проб за {search_time:.0f} с")
print()
print(f"MAE на кросс-валидации по умолчанию:  {default_cv:.2f}")
print(f"MAE на кросс-валидации у лучшей пробы: {study.best_value:.2f}")
print()
print("лучшие параметры:")
for name, value in study.best_params.items():
    print(f"   {name:20} {value:.4g}")

In [ ]:
history = [t.value for t in study.trials]
running_best = np.minimum.accumulate(history)
importance = optuna.importance.get_param_importances(study)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(range(1, len(history) + 1), history, color="#6B7484",
                s=35, label="проба")
axes[0].plot(range(1, len(history) + 1), running_best, color="#2B5CE6",
             lw=2, label="лучшее на данный момент")
axes[0].axhline(default_cv, ls="--", color="#C0393B",
                label="параметры по умолчанию")
axes[0].set_xlabel("номер пробы")
axes[0].set_ylabel("MAE на кросс-валидации, тыс. $")
axes[0].set_title("Ход оптимизации")
axes[0].legend()

names = list(importance.keys())[::-1]
values = [importance[n] for n in names]
axes[1].barh(names, values, color="#2B5CE6", alpha=0.85)
axes[1].set_xlabel("вклад в изменение качества")
axes[1].set_title("Какие параметры вообще важны")

plt.tight_layout()
plt.show()

print("важность параметров:")
for name, value in importance.items():
    print(f"   {name:20} {value:.2f}")

**Левый график.** Точки не ползут вниз подряд: Optuna продолжает пробовать и далёкие варианты, иначе застрянет в первом же приличном. Синяя линия — лучшее, что найдено к этому моменту.

**Правый график.** Больше всего на качество влияет скорость обучения, остальные параметры — заметно слабее. Такой график полезен сам по себе: в следующий раз бюджет стоит тратить на важные параметры, а остальные оставить по умолчанию.

In [ ]:
# Обучаем финальную модель найденными параметрами на всей обучающей
# выборке — и только теперь смотрим на тест
tuned = HistGradientBoostingRegressor(
    max_iter=2000, early_stopping=True, n_iter_no_change=50,
    validation_fraction=0.2, scoring="neg_mean_absolute_error",
    random_state=RANDOM_STATE, **study.best_params,
).fit(X_train, y_train)

pred_boost = boost.predict(X_test)
pred_tuned = tuned.predict(X_test)
mae_tuned = mean_absolute_error(y_test, pred_tuned)
scores["бустинг + Optuna"] = mae_tuned

# Надёжна ли разница? Сравним ошибки двух моделей на каждом объекте теста:
# среднее этих разностей и есть выигрыш, а его стандартная ошибка
# показывает, насколько он мог получиться случайно
gain = np.abs(y_test - pred_boost) - np.abs(y_test - pred_tuned)
margin = 2 * gain.std() / np.sqrt(len(gain))

print(f"{'':26} {'кросс-валидация':>16} {'тест':>8}")
print(f"{'бустинг по умолчанию':26} {default_cv:16.2f} {mae_boost:8.2f}")
print(f"{'бустинг после Optuna':26} {study.best_value:16.2f} {mae_tuned:8.2f}")
print(f"{'выигрыш':26} {default_cv - study.best_value:+16.2f} {gain.mean():+8.2f}")
print()
print(f"выигрыш на тесте: {gain.mean():+.2f} ± {margin:.2f} тыс. $")
print(f"интервал от {gain.mean() - margin:+.2f} до {gain.mean() + margin:+.2f}"
      + (" — включает ноль" if abs(gain.mean()) < margin else ""))

На кросс-валидации подбор выиграл около полутысячи долларов, а на тесте выигрыш почти исчез, и интервал включает ноль. Это не значит, что Optuna не работает. Дело в двух вещах, и обе стоит запомнить до соревнования.

**Выигрыш на той выборке, где подбирали, всегда завышен.** Из тридцати проб мы выбрали лучшую — часть её преимущества просто везение на этих фолдах. На новых данных везение не повторяется.

**Десятые доли не видны на одной тестовой выборке.** Разброс оценки на пяти тысячах районов — около трёхсот долларов в обе стороны. Разница меньше этого неотличима от случайности.

В соревновании это выглядит как разрыв между публичным и приватным лидербордом: подогнанное под видимую часть теста улучшение на скрытой части пропадает. Поэтому порядок работы такой: **сначала данные и признаки, потом модель, и только в конце — подбор параметров**. Переход от дерева к бустингу дал здесь больше пятнадцати тысяч долларов, подбор — доли тысячи.

---
## 7. Библиотеки бустинга

`HistGradientBoosting` из scikit-learn есть всегда и устроен как LightGBM. На практике чаще берут отдельные библиотеки — у них больше возможностей: своя обработка категорий, GPU, разные функции потерь.

In [ ]:
available = {}

# Ловим Exception, а не ImportError: библиотека может установиться,
# но не загрузиться — например, без системного OpenMP на macOS
for name, importer in [
    ("XGBoost", lambda: __import__("xgboost").XGBRegressor),
    ("LightGBM", lambda: __import__("lightgbm").LGBMRegressor),
    ("CatBoost", lambda: __import__("catboost").CatBoostRegressor),
]:
    try:
        available[name] = importer()
    except Exception as e:
        available[name] = None
        print(f"{name} недоступен ({type(e).__name__}), пропускаем")

bench = []

def measure(name, model):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    mae = mean_absolute_error(y_test, model.predict(X_test))
    bench.append({"библиотека": name, "MAE": round(mae, 2),
                  "время, с": round(elapsed, 1)})
    print(f"{name:32} MAE {mae:6.2f}   {elapsed:5.1f} с")

print()
measure("HistGradientBoosting (sklearn)",
        HistGradientBoostingRegressor(max_iter=500, learning_rate=0.1,
                                      random_state=RANDOM_STATE))
if available["XGBoost"]:
    measure("XGBoost", available["XGBoost"](
        n_estimators=500, learning_rate=0.1, max_depth=6,
        n_jobs=-1, random_state=RANDOM_STATE))
if available["LightGBM"]:
    measure("LightGBM", available["LightGBM"](
        n_estimators=500, learning_rate=0.1, num_leaves=31,
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1))
if available["CatBoost"]:
    measure("CatBoost", available["CatBoost"](
        iterations=500, learning_rate=0.1, depth=6,
        random_seed=RANDOM_STATE, verbose=0))

print()
display(pd.DataFrame(bench).sort_values("MAE"))

**Разница в качестве между библиотеками небольшая — намного меньше, чем между одним деревом и любым из бустингов. А время обучения может различаться в разы.** Выбирать стоит по удобству и скорости, а не гоняться за третьим знаком после запятой: сэкономленное время лучше потратить на новые признаки.

---
## 8. Стекинг

Последний способ собрать ансамбль — обучить несколько разных моделей и доверить ещё одной решать, кого слушать.

In [ ]:
stack = StackingRegressor(
    estimators=[
        ("ridge", make_pipeline(StandardScaler(), RidgeCV())),
        ("forest", RandomForestRegressor(n_estimators=150, n_jobs=-1,
                                         random_state=RANDOM_STATE)),
        ("boost", HistGradientBoostingRegressor(max_iter=300,
                                                random_state=RANDOM_STATE)),
    ],
    final_estimator=RidgeCV(),
    cv=3,          # предсказания out-of-fold, иначе будет утечка
    n_jobs=-1,
)

start = time.perf_counter()
stack.fit(X_train, y_train)
stack_time = time.perf_counter() - start

mae_stack = mean_absolute_error(y_test, stack.predict(X_test))
scores["стекинг"] = mae_stack

ridge_alone = make_pipeline(StandardScaler(), RidgeCV()).fit(X_train, y_train)
scores["линейная модель"] = mean_absolute_error(y_test, ridge_alone.predict(X_test))

# Участники после обучения стекинга лежат в named_estimators_ —
# их качество по отдельности можно посмотреть бесплатно
print(f"{'участник':10} {'MAE на тесте':>13} {'вес у мета-модели':>19}")
for (name, member), weight in zip(stack.named_estimators_.items(),
                                  stack.final_estimator_.coef_):
    member_mae = mean_absolute_error(y_test, member.predict(X_test))
    print(f"{name:10} {member_mae:13.2f} {weight:+19.3f}")
print(f"{'стекинг':10} {mae_stack:13.2f}")
print()
print(f"обучение стекинга: {stack_time:.1f} с")
print()
print("Мета-модель почти весь вес отдала бустингу, а линейной модели — около")
print("нуля: она сама по себе слабая и ничего не добавляет к остальным.")
print("Параметр cv=3 здесь принципиален: без него мета-модель училась бы")
print("на предсказаниях для объектов, которые базовые модели уже видели.")

---
## Итоговое сравнение

Собираем всё, что обучили за занятие.

In [ ]:
table = (pd.DataFrame({"модель": list(scores), "MAE": list(scores.values())})
         .sort_values("MAE", ascending=False))

plt.figure(figsize=(10, 5))
plt.barh(table["модель"], table["MAE"], color="#2B5CE6", alpha=0.85)
for name, value in zip(table["модель"], table["MAE"]):
    plt.text(value + 1, name, f"{value:.1f}", va="center")
plt.xlabel("MAE на тесте, тыс. $  (меньше — лучше)")
plt.title("Путь от константы до ансамбля")
plt.xlim(0, max(table["MAE"]) * 1.12)
plt.tight_layout()
plt.show()

print(f"ошибка упала с {scores['константа']:.1f} до {table['MAE'].min():.1f} тыс. $ — "
      f"в {scores['константа'] / table['MAE'].min():.1f} раза")
print()
print("Шаги по вкладу: одно дерево вместо константы — главный скачок,")
print("ансамбль вместо дерева — второй по величине. Всё остальное")
print("(стекинг, подбор параметров) двигает результат на десятые доли,")
print("и на одной тестовой выборке эти движения неотличимы от шума.")

---
## Итоги занятия

- Ансамбль работает потому, что ошибки его участников направлены в разные стороны: **ни одно дерево внутри бэггинга не дотягивает до их общего результата**
- В бэггинг берут переобученные модели — он лечит разброс. В бустинг слабые — он лечит смещение
- Случайный лес почти не требует настройки и даёт бесплатную out-of-bag оценку
- **Ранняя остановка обязательна для бустинга:** лишние деревья здесь не нейтральны, а вредны
- Скорость обучения и число деревьев связаны обратно, но качество от скорости почти не зависит — она определяет бюджет
- Optuna подбирает параметры по результатам предыдущих проб и сразу показывает, какие из них вообще на что-то влияют
- **Выигрыш на выборке, где подбирали, завышен.** Улучшение в десятые доли на одной тестовой выборке не отличить от случайности
- Разница между библиотеками бустинга мала; стекинг и подбор параметров дают доли тысячи долларов там, где переход от дерева к ансамблю дал больше десяти тысяч

## Домашнее задание 3

Выдаётся сегодня: **соревнование на Kaggle** — предсказание цены по объявлению о продаже автомобиля. Балл зависит от позиции в итоговом лидерборде.

Всё, что было на занятии, там понадобится: бустинг с ранней остановкой, честная валидация, подбор параметров. Но помните вывод раздела 6 — **признаки дают больше, чем настройка**, а финальную отправку выбирайте по своей кросс-валидации, а не по публичному месту.

Вместе с ним выдаётся **домашнее задание 4** — итоговый проект с дедлайном в сессию.